# Lyapunov Smoke Campaign Characterization

Loads the completed `N20x40_nsh1_a1-1_S10_cycles20_smoke` campaign and produces Lyapunov, Chern, filling-fraction, and spatial charge-profile diagnostics. The Lyapunov gap proxy used here is `min_i |lambda_i|` for each sample and cycle.


## Setup
CPU allocation, paths, plotting style, and campaign labels.


In [ ]:
from __future__ import annotations

import os

CPU_AFFINITY_ENABLED = False
CPU_START = 0
CPU_STOP = 8  # exclusive

if CPU_AFFINITY_ENABLED:
    cpus = set(range(int(CPU_START), int(CPU_STOP)))
    if hasattr(os, "sched_setaffinity"):
        os.sched_setaffinity(0, cpus)
        print(f"Pinned notebook process to CPUs {min(cpus)}..{max(cpus)}")
    else:
        print("CPU affinity is not supported on this platform.")
else:
    print("CPU affinity unchanged.")


In [ ]:
import json
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd

CAMPAIGN_ID = "N20x40_nsh1_a1-1_S10_cycles20_smoke"
DPI = 300


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "colab_lyapunov" / "gpu_data").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate repository root containing colab_lyapunov/gpu_data")


PROJECT_ROOT = find_project_root()
CAMPAIGN_ROOT = PROJECT_ROOT / "colab_lyapunov" / "gpu_data" / "lyapunov_spectra" / "campaigns" / CAMPAIGN_ID

available_fonts = {font.name for font in font_manager.fontManager.ttflist}
font_family = "CMU Sans Serif" if "CMU Sans Serif" in available_fonts else "DejaVu Sans"
mpl.rcParams.update({
    "font.family": font_family,
    "font.size": 6,
    "axes.titlesize": 6,
    "axes.labelsize": 6,
    "xtick.labelsize": 5,
    "ytick.labelsize": 5,
    "legend.fontsize": 5,
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
    "axes.linewidth": 0.7,
    "lines.linewidth": 1.2,
})

CONFIG_ORDER = [
    "N20x40_DW0_dwtrunc0_a2-1_nsh1_perfect_correction",
    "N20x40_DW1_dwtrunc1_a2-30_nsh1_perfect_correction",
    "N20x40_DW0_dwtrunc0_a2-1_nsh1_postselect",
    "N20x40_DW1_dwtrunc1_a2-30_nsh1_postselect",
]
CONFIG_LABELS = {
    "N20x40_DW0_dwtrunc0_a2-1_nsh1_postselect": "DW0 postselect",
    "N20x40_DW0_dwtrunc0_a2-1_nsh1_perfect_correction": "DW0 perfect correction",
    "N20x40_DW1_dwtrunc1_a2-30_nsh1_postselect": "DW1 postselect",
    "N20x40_DW1_dwtrunc1_a2-30_nsh1_perfect_correction": "DW1 perfect correction",
}
COLORS = dict(zip(CONFIG_ORDER, mpl.colormaps["tab10"].colors[:4]))
SNAPSHOT_CYCLES = [5, 10, 15, 20]


print(f"project root: {PROJECT_ROOT}")
print(f"campaign root: {CAMPAIGN_ROOT}")
print(f"font: {font_family}")


## Data Loading and Derived Observables
Load all four runs, validate shapes, and derive filling fraction, Lyapunov gap proxy, and spatial charge profiles.


In [ ]:
run_index = pd.read_csv(CAMPAIGN_ROOT / "run_index.csv")
runs = {}
for config_id in CONFIG_ORDER:
    run_dir = CAMPAIGN_ROOT / "runs" / config_id
    summary = json.loads((run_dir / "run_summary.json").read_text())
    with np.load(run_dir / "lyapunov_spectra.npz", allow_pickle=False) as data:
        spectra = data["lyapunov_spectra"]
        normalization = str(data["normalization"].tolist())
    with np.load(run_dir / "real_space_chern.npz", allow_pickle=False) as data:
        chern = data["real_space_chern"]
    with np.load(run_dir / "local_charge_cell.npz", allow_pickle=False) as data:
        charge = data["local_charge_cell"]
    nx = charge.shape[2]
    ny = charge.shape[3]
    total_charge = charge.sum(axis=(-2, -1))
    runs[config_id] = {
        "summary": summary,
        "spectra": spectra,
        "chern": chern,
        "charge": charge,
        "normalization": normalization,
        "total_charge": total_charge,
        "filling_fraction": total_charge / float(nx * ny),
        "x_profile": charge.mean(axis=-1),
        "lyapunov_min_abs": np.min(np.abs(spectra), axis=-1),
    }

assert set(runs) == set(CONFIG_ORDER)
for config_id, run in runs.items():
    summary = run["summary"]
    spectra = run["spectra"]
    chern = run["chern"]
    charge = run["charge"]
    filling_fraction = run["filling_fraction"]
    lyapunov_min_abs = run["lyapunov_min_abs"]
    samples = int(summary["actual_samples"])
    cycles = int(summary["cycles"])
    n_vec = int(summary["lyapunov_nvec"])
    assert spectra.shape == (samples, cycles, n_vec), (config_id, spectra.shape)
    assert chern.shape == (samples, cycles), (config_id, chern.shape)
    assert charge.shape == (samples, cycles, 20, 40), (config_id, charge.shape)
    assert filling_fraction.shape == (samples, cycles), (config_id, filling_fraction.shape)
    assert lyapunov_min_abs.shape == (samples, cycles), (config_id, lyapunov_min_abs.shape)
    assert np.isfinite(spectra).all(), config_id
    assert np.isfinite(chern).all(), config_id
    assert np.isfinite(charge).all(), config_id
    assert np.isfinite(filling_fraction).all(), config_id
    assert np.isfinite(lyapunov_min_abs).all(), config_id
    if summary["protocol"] == "postselect":
        assert samples == 1, config_id
    if summary["protocol"] == "perfect_correction":
        assert samples == 10, config_id

max_cycles = min(run["spectra"].shape[1] for run in runs.values())
assert all(1 <= c <= max_cycles for c in SNAPSHOT_CYCLES)

summary_rows = []
for config_id in CONFIG_ORDER:
    s = runs[config_id]["summary"]
    summary_rows.append({
        "config": CONFIG_LABELS[config_id],
        "samples": s["actual_samples"],
        "cycles": s["cycles"],
        "n_vec": s["lyapunov_nvec"],
        "DW": s["DW"],
        "alpha_2": s["alpha_2"],
        "normalization": runs[config_id]["normalization"],
    })
pd.DataFrame(summary_rows)


In [ ]:
def sample_mean_stderr(a: np.ndarray, axis: int = 0) -> tuple[np.ndarray, np.ndarray]:
    mean = np.mean(a, axis=axis)
    n = a.shape[axis]
    if n > 1:
        err = np.std(a, axis=axis, ddof=1) / np.sqrt(float(n))
    else:
        err = np.zeros_like(mean)
    return mean, err


def plot_grid_labels(axes, ylabel_prefix: str = "cycle") -> None:
    for row, cycle in enumerate(SNAPSHOT_CYCLES):
        axes[row, 0].set_ylabel(f"{ylabel_prefix} {cycle}")
    for col, config_id in enumerate(CONFIG_ORDER):
        axes[0, col].set_title(CONFIG_LABELS[config_id])


cycle_numbers = np.arange(1, max_cycles + 1)


## Lyapunov Spectrum Characterization
Histograms, sample-averaged spectra, and the finite-size gap proxy $\min_i |\lambda_i|$.


In [ ]:
lyap_values = np.concatenate([
    runs[cfg]["spectra"][:, cycle - 1, :].ravel()
    for cfg in CONFIG_ORDER
    for cycle in SNAPSHOT_CYCLES
])
lyap_bins = np.linspace(float(lyap_values.min()), float(lyap_values.max()), 81)

fig, axes = plt.subplots(4, 4, figsize=(5.5, 4.0), constrained_layout=True)
for col, config_id in enumerate(CONFIG_ORDER):
    color = COLORS[config_id]
    for row, cycle in enumerate(SNAPSHOT_CYCLES):
        ax = axes[row, col]
        vals = runs[config_id]["spectra"][:, cycle - 1, :].ravel()
        ax.hist(vals, bins=lyap_bins, color=color, alpha=0.82, linewidth=0)
        ax.text(0.98, 0.88, f"S={runs[config_id]['spectra'].shape[0]}", transform=ax.transAxes, ha="right", va="top")
        if row == len(SNAPSHOT_CYCLES) - 1:
            ax.set_xlabel(r"$\lambda$")
plot_grid_labels(axes)
fig.suptitle("Lyapunov spectra histograms", y=1.02)
plt.show()


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(5.5, 4.0), constrained_layout=True, sharex=True, sharey=True)
for col, config_id in enumerate(CONFIG_ORDER):
    color = COLORS[config_id]
    for row, cycle in enumerate(SNAPSHOT_CYCLES):
        ax = axes[row, col]
        mean_spec = np.sort(np.mean(runs[config_id]["spectra"][:, cycle - 1, :], axis=0))
        ax.plot(np.arange(mean_spec.size), np.abs(mean_spec), color=color, linestyle=" ", marker=".", markersize=1)
        ax.set_yscale("log")
        if row == len(SNAPSHOT_CYCLES) - 1:
            ax.set_xlabel("ordered index")
plot_grid_labels(axes)
fig.suptitle(r"Sample-averaged Lyapunov spectra $|\lambda_i|$", y=1.02)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(2.75, 1.8), constrained_layout=True)
for config_id in CONFIG_ORDER:
    min_abs = runs[config_id]["lyapunov_min_abs"]
    mean, err = sample_mean_stderr(min_abs, axis=0)
    color = COLORS[config_id]
    ax.plot(cycle_numbers, mean, color=color, label=f"{CONFIG_LABELS[config_id]} (S={min_abs.shape[0]})")
    ax.fill_between(cycle_numbers, mean - err, mean + err, color=color, alpha=0.16, linewidth=0)
ax.set_xlabel("cycle")
ax.set_ylabel(r"$\min_i |\lambda_i|$")
ax.set_yscale("log")
ax.set_title(r"Lyapunov gap proxy: $\min_i |\lambda_i|$")
ax.legend(ncol=1, frameon=False)
plt.show()


## Real-Space Chern Characterization
Snapshot histograms and sample-averaged Chern number versus cycle.


In [ ]:
chern_values = np.concatenate([
    runs[cfg]["chern"][:, cycle - 1].ravel()
    for cfg in CONFIG_ORDER
    for cycle in SNAPSHOT_CYCLES
])
pad = 0.02 * max(1.0, float(chern_values.max() - chern_values.min()))
chern_bins = np.linspace(float(chern_values.min() - pad), float(chern_values.max() + pad), 13)

fig, axes = plt.subplots(4, 4, figsize=(5.5, 4.0), constrained_layout=True)
for col, config_id in enumerate(CONFIG_ORDER):
    color = COLORS[config_id]
    for row, cycle in enumerate(SNAPSHOT_CYCLES):
        ax = axes[row, col]
        vals = runs[config_id]["chern"][:, cycle - 1]
        if vals.size == 1:
            ax.axvline(vals[0], color=color, lw=1.8)
            ax.plot(vals[0], 1.0, marker="o", color=color, ms=3)
        else:
            ax.hist(vals, bins=chern_bins, color=color, alpha=0.82, linewidth=0)
        if row == len(SNAPSHOT_CYCLES) - 1:
            ax.set_xlabel(r"$C$")
plot_grid_labels(axes)
fig.suptitle("Real-space Chern number histograms", y=1.02)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(2.75, 1.8), constrained_layout=True)
for config_id in CONFIG_ORDER:
    chern = runs[config_id]["chern"]
    mean, err = sample_mean_stderr(chern, axis=0)
    color = COLORS[config_id]
    ax.plot(cycle_numbers, np.abs(mean-1), color=color, label=f"{CONFIG_LABELS[config_id]} (S={chern.shape[0]})")
    #ax.fill_between(cycle_numbers, mean - err, mean + err, color=color, alpha=0.16, linewidth=0)
ax.set_xlabel("cycle")
ax.set_ylabel(r"$|C-1|$")
ax.set_yscale("log")
ax.set_title("Sample-averaged real-space Chern number")
ax.legend(loc="best")
plt.show()


## Filling-Fraction Characterization
Snapshot histograms, sample-averaged filling fraction, and sample-to-sample filling fluctuations.


In [ ]:
filling_values = np.concatenate([
    runs[cfg]["filling_fraction"][:, cycle - 1].ravel()
    for cfg in CONFIG_ORDER
    for cycle in SNAPSHOT_CYCLES
])
pad = 0.02 * max(0.01, float(filling_values.max() - filling_values.min()))
filling_bins = np.linspace(float(filling_values.min() - pad), float(filling_values.max() + pad), 13)

fig, axes = plt.subplots(4, 4, figsize=(5.5, 4.0), constrained_layout=True)
for col, config_id in enumerate(CONFIG_ORDER):
    color = COLORS[config_id]
    for row, cycle in enumerate(SNAPSHOT_CYCLES):
        ax = axes[row, col]
        vals = runs[config_id]["filling_fraction"][:, cycle - 1]
        if vals.size == 1:
            ax.axvline(vals[0], color=color, lw=1.8)
            ax.plot(vals[0], 1.0, marker="o", color=color, ms=3)
        else:
            ax.hist(vals, bins=filling_bins, color=color, alpha=0.82, linewidth=0)
        if row == len(SNAPSHOT_CYCLES) - 1:
            ax.set_xlabel(r"$\nu = Q_{\rm tot}/(N_x N_y)$")
plot_grid_labels(axes)
fig.suptitle("Filling-fraction histograms", y=1.02)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(5.5, 1.6), sharey=True, constrained_layout=True)
for ax, config_id in zip(axes, CONFIG_ORDER):
    filling_fraction = runs[config_id]["filling_fraction"]
    mean, err = sample_mean_stderr(filling_fraction, axis=0)
    color = COLORS[config_id]
    ax.plot(cycle_numbers, mean, color=color)
    ax.fill_between(cycle_numbers, mean - err, mean + err, color=color, alpha=0.18, linewidth=0)
    ax.set_title(f"{CONFIG_LABELS[config_id]}\nS={filling_fraction.shape[0]}")
    ax.set_xlabel("cycle")
axes[0].set_ylabel(r"$\nu = Q_{\rm tot}/(N_x N_y)$")
fig.suptitle("Sample-averaged filling fraction", y=1.08)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(2.75, 1.8), constrained_layout=True)
for config_id in CONFIG_ORDER:
    filling_fraction = runs[config_id]["filling_fraction"]
    if filling_fraction.shape[0] > 1:
        std = np.std(filling_fraction, axis=0, ddof=1)
    else:
        std = np.zeros(filling_fraction.shape[1], dtype=float)
    color = COLORS[config_id]
    ax.plot(cycle_numbers, std, color=color, label=f"{CONFIG_LABELS[config_id]} (S={filling_fraction.shape[0]})")
ax.set_xlabel("cycle")
ax.set_ylabel(r"sample std. of $\nu$")
ax.set_title("Filling-fraction sample-to-sample standard deviation")
ax.legend(ncol=1, frameon=False)
plt.show()


## Spatial Charge-Profile Characterization
Sample-averaged fixed-$x$, $y$-averaged charge profiles versus cycle.


In [ ]:
nx = next(iter(runs.values()))["charge"].shape[2]
x_values = np.arange(nx // 2)
cmap = mpl.colormaps["tab10"]
colors_by_x = {int(x): cmap(i % 10) for i, x in enumerate(x_values)}

fig, axes = plt.subplots(2, 2, figsize=(5.5, 3.7), sharey=True)
legend_handles = []
for ax, config_id in zip(axes.ravel(), CONFIG_ORDER):
    x_profile = runs[config_id]["x_profile"][:, :, x_values]
    mean, _ = sample_mean_stderr(x_profile, axis=0)
    dw_loc = runs[config_id]["summary"]["dw_metadata"].get("dw_loc", [])
    for local_i, x in enumerate(x_values):
        color = colors_by_x[int(x)]
        line, = ax.plot(cycle_numbers, mean[:, local_i], color=color, alpha=0.95, lw=0.9, label=f"x={int(x)}")
        if ax is axes.ravel()[0]:
            legend_handles.append(line)
    if dw_loc:
        dw_text = f"DW x={dw_loc}; shown x=0..{x_values[-1]}"
    else:
        dw_text = f"no DW; center x={nx // 2}; shown x=0..{x_values[-1]}"
    ax.text(0.02, 0.96, dw_text, transform=ax.transAxes, ha="left", va="top")
    ax.set_title(f"{CONFIG_LABELS[config_id]} (S={x_profile.shape[0]})")
    ax.set_xlabel("cycle")
    ax.set_ylabel(r"$\langle N_y^{-1}\sum_y Q_{x,y}\rangle_{\rm samples}$")
fig.subplots_adjust(left=0.23, right=0.98, bottom=0.12, top=0.88, wspace=0.28, hspace=0.42)
fig.legend(handles=legend_handles, loc="center left", ncol=1, frameon=False, bbox_to_anchor=(0.03, 0.5))
fig.suptitle(r"Fixed-$x$, $y$-averaged charge profiles", y=1.03)
plt.show()


In [ ]:
nx = next(iter(runs.values()))["charge"].shape[2]
x_values = np.arange(nx // 2)
cmap = mpl.colormaps["tab10"]
colors_by_x = {int(x): cmap(i % 10) for i, x in enumerate(x_values)}

fig, axes = plt.subplots(2, 2, figsize=(5.5, 3.7), sharey=True)
legend_handles = []
for ax, config_id in zip(axes.ravel(), CONFIG_ORDER):
    x_profile = runs[config_id]["x_profile"][:, :, x_values]
    if x_profile.shape[0] > 1:
        std = np.std(x_profile, axis=0, ddof=1)
    else:
        std = np.zeros(x_profile.shape[1:], dtype=float)
    dw_loc = runs[config_id]["summary"]["dw_metadata"].get("dw_loc", [])
    for local_i, x in enumerate(x_values):
        color = colors_by_x[int(x)]
        line, = ax.plot(cycle_numbers, std[:, local_i], color=color, alpha=0.95, lw=0.9, label=f"x={int(x)}")
        if ax is axes.ravel()[0]:
            legend_handles.append(line)
    if dw_loc:
        dw_text = f"DW x={dw_loc}; shown x=0..{x_values[-1]}"
    else:
        dw_text = f"no DW; center x={nx // 2}; shown x=0..{x_values[-1]}"
    ax.text(0.02, 0.96, dw_text, transform=ax.transAxes, ha="left", va="top")
    ax.set_title(f"{CONFIG_LABELS[config_id]} (S={x_profile.shape[0]})")
    ax.set_xlabel("cycle")
    ax.set_ylabel(r"$\operatorname{std}_{\rm samples}[N_y^{-1}\sum_y Q_{x,y}]$")
fig.subplots_adjust(left=0.23, right=0.98, bottom=0.12, top=0.88, wspace=0.28, hspace=0.42)
fig.legend(handles=legend_handles, loc="center left", ncol=1, frameon=False, bbox_to_anchor=(0.03, 0.5))
fig.suptitle(r"Sample standard deviation of fixed-$x$, $y$-averaged charge", y=1.03)
plt.show()
